# 05 — Evaluación final: calidad de datos, comparación equivalente de modelos y significancia
**CRISP-DM: Evaluación** · Etapa final · Título provisional: *Predicción del precio de productos agrícolas en mercados de El Salvador*

Este notebook **no recalcula** los modelos: lee los resultados que generan los scripts de `scripts/` (carpeta `reports/etapa3/`) y los presenta con tablas y gráficos. Así se ejecuta en segundos y las cifras coinciden con las del informe.

Para generar los resultados desde cero, con el entorno virtual activo y desde la raíz del repositorio:

```bash
python scripts/run_all.py
```

**Secciones:** 1) datos y calidad · 2) nulos y valores atípicos · 3) referencias ingenuas · 4) resultados por producto · 5) ensamble · 6) significancia e intervalos · 7) variables externas (lluvia) · 8) LSTM · 9) conclusiones y limitaciones.

In [ ]:
import sys, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
warnings.filterwarnings("ignore")

RAIZ = Path.cwd().parent                      # el notebook se ejecuta desde notebooks/
sys.path.append(str(RAIZ / "src"))
from prediccion_precios import config

ET = RAIZ / "reports" / "etapa3"              # resultados de los scripts
FIG = RAIZ / "reports" / "figures"            # figuras para el informe
FIG.mkdir(parents=True, exist_ok=True)
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
sns.set_theme(style="whitegrid", context="notebook")
ORDEN = ["ARROZ", "MAÍZ", "FRIJOL", "NARANJA", "TOMATE"]


def leer(nombre):
    ruta = ET / nombre
    if not ruta.exists():
        raise FileNotFoundError(f"Falta {ruta}. Genera los resultados con: python scripts/run_all.py")
    return pd.read_csv(ruta)


def guardar(fig, nombre):
    fig.savefig(FIG / nombre, dpi=200, bbox_inches="tight")
    print("Figura guardada en", FIG / nombre)

## 1. Datos y calidad de la información
La variable objetivo (precio diario del MAG) limita todo el estudio: las demás fuentes llegan a 2024 o más, pero sin precios no pueden aprovecharse.

In [ ]:
cob = leer("cobertura_fuentes.csv")
display(cob)

In [ ]:
aud = leer("auditoria_precios.csv").set_index("producto").loc[ORDEN]
display(aud[["registros", "registros_nd", "dias_con_dato", "primer_dia", "ultimo_dia",
             "semanas_con_dato", "semanas_con_1_o_2_dias"]])
sem = leer("semanas_sin_dato.csv")
print("Semanas sin ningún precio (cierre en domingo) y productos afectados:")
display(sem.groupby("semana_cierre").size().rename("productos_sin_dato").to_frame())

### 1.1 Precios repetidos: el caso de la naranja

In [ ]:
cols = {"pct_identico_2021": "2021", "pct_identico_2022": "2022", "pct_identico_2023": "2023", "pct_identico_2024": "2024"}
rep = aud[list(cols)].rename(columns=cols)
fig, ax = plt.subplots(figsize=(7, 3.6))
sns.heatmap(rep, annot=True, fmt=".1f", cmap="Reds", vmin=0, vmax=100, cbar_kws={"label": "% de días"}, ax=ax)
ax.set_title("Días con precio idéntico al día anterior (%)"); ax.set_ylabel(""); ax.set_xlabel("")
guardar(fig, "etapa3_precios_repetidos.png"); plt.show()

### 1.2 Recorte por IQR del pipeline original
El recorte original (Q1 − 1.5·IQR, Q3 + 1.5·IQR con toda la muestra) solo afecta al **tomate**.

In [ ]:
display(aud[["limite_IQR_superior", "precio_max_crudo", "dias_recortados_IQR"]])

## 2. Nulos y valores atípicos: ¿cambia la conclusión?
Cuatro variantes del tratamiento, evaluadas contra el precio semanal **crudo** en las mismas semanas de prueba: **V0** original, **V1** sin recorte, **V2** causal sin recorte y **V3** causal con recorte calculado solo con entrenamiento.

In [ ]:
sen = leer("sensibilidad_nulos.csv")
obs = sen[sen.subconjunto == "real_observado"]
glob = obs[obs.producto == "GLOBAL"].pivot(index="variante", columns="modelo", values="MAPE_%")
tom = obs[obs.producto == "TOMATE"].pivot(index="variante", columns="modelo", values="MAPE_%")
display(pd.concat({"Global": glob, "Tomate": tom}, axis=1).round(2))

fig, ax = plt.subplots(figsize=(6.5, 3.6))
glob.plot(kind="bar", ax=ax, rot=0, color=["#4C72B0", "#DD8452"])
ax.set_ylabel("MAPE global (%)"); ax.set_xlabel(""); ax.set_title("Sensibilidad al tratamiento de nulos y atípicos")
ax.legend(title="")
guardar(fig, "etapa3_sensibilidad_nulos.png"); plt.show()
print("Diferencia máxima entre variantes (puntos): persistencia %.2f, regresión lineal %.2f"
      % (glob["persistencia"].max() - glob["persistencia"].min(),
         glob["regresion_lineal"].max() - glob["regresion_lineal"].min()))

## 3. Referencias ingenuas y series de tiempo clásicas
Un MAPE solo es «bueno» respecto a una referencia. Sobre las 157 filas de prueba del proyecto:

In [ ]:
m = leer("metricas_baselines.csv")
mg = m[m.producto == "GLOBAL"].pivot(index="metodo", columns="horizonte", values="MAPE_%").round(2)
display(mg.sort_values(1))
fig, ax = plt.subplots(figsize=(7, 3.8))
for met, fila in mg.iterrows():
    ax.plot(fila.index, fila.values, marker="o", label=met, lw=2.4 if met == "persistencia" else 1.2)
ax.set_xlabel("Horizonte (semanas)"); ax.set_ylabel("MAPE global (%)"); ax.set_xticks([1, 2, 4, 8])
ax.set_title("La persistencia es la referencia a vencer"); ax.legend(fontsize=8)
guardar(fig, "etapa3_baselines_horizonte.png"); plt.show()

## 4. Resultados por producto y horizonte
El error de la persistencia cambia por casi dos órdenes de magnitud entre productos, por eso se presenta **por producto**.

In [ ]:
t = leer("tabla_por_producto.csv")
pers = t[t.modelo == "persistencia"].pivot(index="producto", columns="horizonte", values="MAPE").loc[ORDEN].round(2)
display(pers.rename_axis("MAPE (%) de la persistencia"))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2), sharey=True)
for ax, h in zip(axes, (1, 4)):
    r = (t[(t.horizonte == h) & (t.modelo != "persistencia")]
         .pivot(index="modelo", columns="producto", values="razon_vs_persistencia")[ORDEN])
    sns.heatmap(r.clip(upper=3), annot=r.map(lambda v: f"{v:.2f}"), fmt="", cmap="RdBu_r", center=1, vmin=0.5, vmax=1.5,
                cbar=ax is axes[1], ax=ax)
    ax.set_title(f"Razón MAE modelo / persistencia, h = {h} sem"); ax.set_ylabel(""); ax.set_xlabel("")
fig.text(0.5, -0.02, "Menor que 1 (azul) = mejor que la persistencia. El color se trunca en 1.5 (el arroz llega a 15).",
         ha="center", fontsize=9)
guardar(fig, "etapa3_razon_por_producto.png"); plt.show()

In [ ]:
sin = t[(t.producto != "ARROZ") & (t.modelo != "persistencia")]
prom = sin.groupby(["modelo", "horizonte"]).razon_vs_persistencia.mean().unstack().round(3)
display(prom.rename_axis("Razón promedio entre productos (sin arroz)"))

## 5. Ensamble contraído hacia la persistencia
Pronóstico = p<sub>t</sub> · (1 + w · r), con r el promedio de las variaciones que predicen Ridge, Random Forest y Gradient Boosting, y w ∈ {0, .25, .5, .75, 1} elegido por producto y horizonte con validación previa a la prueba (w = 0 es la persistencia).

In [ ]:
ens = leer("ensamble_resumen.csv")
e = ens[ens.modelo == "ensamble_contraido"].pivot(index="producto", columns="horizonte", values="razon_vs_persistencia").loc[ORDEN].round(3)
display(e.rename_axis("Razón MAE ensamble / persistencia"))
w = leer("ensamble_pesos.csv").pivot(index="producto", columns="horizonte", values="w").loc[ORDEN]
display(w.rename_axis("Peso w elegido (0 = persistencia)"))

## 6. Significancia estadística e intervalos de predicción
Prueba de Diebold-Mariano (error absoluto escalado por producto, 28 semanas, Newey-West con corrección de Harvey et al.): H0 = igual precisión que la persistencia.

In [ ]:
dm = leer("diebold_mariano.csv")
p = dm.pivot(index="modelo", columns="horizonte", values="p_valor")
display(p.round(3).rename_axis("Valor p (frente a la persistencia)"))
dif = dm.pivot(index="modelo", columns="horizonte", values="dif_media_escalada").round(2)
display(dif.rename_axis("Diferencia media escalada (> 0 = el modelo es peor)"))

In [ ]:
iv = leer("intervalos_persistencia.csv")
c80 = iv[iv.nivel_nominal == 0.8].pivot(index="producto", columns="horizonte", values="cobertura_prueba").loc[ORDEN]
fig, ax = plt.subplots(figsize=(6.5, 3.8))
sns.heatmap(c80, annot=True, fmt=".2f", cmap="RdBu", center=0.8, vmin=0.2, vmax=1.0, ax=ax,
            cbar_kws={"label": "cobertura real"})
ax.set_title("Cobertura en la prueba del intervalo nominal de 80 %"); ax.set_xlabel("Horizonte (semanas)"); ax.set_ylabel("")
guardar(fig, "etapa3_cobertura_intervalos.png"); plt.show()
print("Cobertura media real:", iv.groupby("nivel_nominal").cobertura_prueba.mean().round(3).to_dict(), "(nominal 0.8 y 0.9)")

## 7. Variables externas: ¿aporta la lluvia mensual?
Cuatro conjuntos de variables con los modelos afinados y el mismo protocolo. La variante «mensual del mes en curso» incluye días futuros del mes: es una cota optimista, no utilizable en la práctica.

In [ ]:
ll = leer("prueba_lluvia_mensual.csv")
orden_v = ["base", "sin_lluvia", "+mensual_prev", "+mensual_act"]
tabla = (ll[ll.producto != "ARROZ"].groupby(["variante", "modelo"]).razon.mean().unstack(0)[orden_v].round(3))
display(tabla.rename_axis("Razón MAE / persistencia (sin arroz, promedio de horizontes)"))
print("Promedio de todos los modelos:", tabla.mean().round(3).to_dict())

## 8. LSTM con el mismo protocolo
Disponible solo si se ejecutó `python scripts/a11_lstm.py` (requiere TensorFlow). Las salidas `*_stub.csv` son de una verificación de flujo de datos y **no** corresponden al LSTM.

In [ ]:
ruta = ET / "lstm_resumen.csv"
if ruta.exists():
    ls = pd.read_csv(ruta)
    r = ls.pivot(index="producto", columns="horizonte", values="razon_vs_persistencia").loc[ORDEN].round(2)
    display(r.rename_axis("Razón MAE LSTM / persistencia"))
    print("Promedio sin arroz:", ls[ls.producto != "ARROZ"].groupby("horizonte").razon_vs_persistencia.mean().round(3).to_dict())
else:
    print("Aún no hay lstm_resumen.csv. Ejecuta: python scripts/a11_lstm.py --precios data/interim/precios_consolidados.xlsx")

## 9. Conclusiones y limitaciones
**Conclusiones**
1. La persistencia es la mejor referencia (MAPE global de 4.49 % a una semana y 16.24 % a ocho) y no la superan ni ETS ni ARIMA.
2. Con afinación equivalente y origen móvil, ningún modelo la supera de forma consistente; Diebold-Mariano los da como significativamente peores.
3. El tratamiento de nulos y atípicos cambia el MAPE en menos de 0.6 puntos: se adopta el causal por rigor, no porque explique el resultado.
4. Un ensamble que corrige la persistencia solo si la validación lo justifica la iguala, pero no la mejora.
5. Ni la lluvia diaria ni la mensual aportan información predictiva.

**Limitaciones:** los precios llegan a junio de 2024; la prueba tiene 28 semanas por producto; los precios de la naranja se repiten en el 87 % de los días; los intervalos empíricos subcubren (63 % frente a 80 % nominal); las fuentes de lluvia no coinciden y la producción de 2023-2024 es imputada o no oficial.